## Vicsek particles inside a sphere

$|\mathbf x|\le a$. Vicsek alignment and cone noise set the preferred velocity
$\mathbf v^*=v_0(\cos\theta\sin\phi,\cos\theta\cos\phi,\sin\theta)$.

The wall acts only on the radial component. In the nearby layer, damp outward motion:
$$\dot{\mathbf v}_{wall}=-k_w w(r)\max(\mathbf v\cdot\hat{\mathbf r},0)\hat{\mathbf r},\qquad
w(r)=\left[\frac{r-a+\delta}{\delta}\right]_0^1.$$
The radial damping uses its exponential step; tangential motion is unchanged. At contact enforce
$$\mathbf v\leftarrow\mathbf v-(\mathbf v\cdot\hat{\mathbf r})\hat{\mathbf r},\qquad \mathbf v\cdot\hat{\mathbf r}=0.$$
No speed renormalization and no invented tangential fallback: purely radial motion becomes zero at contact. Actual speed can therefore be below the preferred $v_0$.

Positions use the current velocity and are projected inside if needed; wall events are resolved at time steps. This strict boundary condition keeps contacted particles on the surface. No periodic boundaries or particle–particle repulsion.


In [ ]:
import numpy as np


In [ ]:
# Initialization (normalized units).
N = 150
a = 5.0               # Sphere radius; center is the origin.
v_0 = 0.3
R = 1.5               # Alignment radius.
eta = 0.6             # Angular-rate width; cone half-angle = min(pi, eta*dt/2).
wall_width = 1.0
wall_strength = 5.0    # Outward radial damping rate (1/time).
dt = 0.05
n_steps = 800
seed = 42


In [ ]:
# Your velocity convention: theta is elevation; phi starts from +y.
def update_velocity_3d(theta, phi, v_0):
    return v_0 * np.stack((np.cos(theta) * np.sin(phi),
                          np.cos(theta) * np.cos(phi),
                          np.sin(theta)), axis=-1)

# Uniform random turning on a spherical cap, while retaining theta/phi state.
def add_direction_noise_3d(average, eta, rng, dt=1.0):
    reference = np.tile([0.0, 0.0, 1.0], (len(average), 1))
    reference[np.abs(average[:, 2]) > 0.9] = [1.0, 0.0, 0.0]
    e1 = np.cross(reference, average)
    e1 /= np.linalg.norm(e1, axis=1)[:, None]
    e2 = np.cross(average, e1)
    alpha = min(np.pi, eta * dt / 2)
    cos_beta = rng.uniform(np.cos(alpha), 1.0, size=len(average))
    sin_beta = np.sqrt(np.maximum(0, 1 - cos_beta**2))
    psi = rng.uniform(0, 2 * np.pi, size=len(average))
    updated = (cos_beta[:, None] * average
               + (sin_beta * np.cos(psi))[:, None] * e1
               + (sin_beta * np.sin(psi))[:, None] * e2)
    return updated / np.linalg.norm(updated, axis=1)[:, None]



In [ ]:
def initialize_sphere(N, a, rng):
    # Uniform volume distribution: radius scales as U**(1/3).
    z = rng.uniform(-1, 1, N)
    azimuth = rng.uniform(0, 2*np.pi, N)
    direction = np.column_stack((np.sqrt(1-z*z)*np.cos(azimuth),
                                 np.sqrt(1-z*z)*np.sin(azimuth), z))
    position = a * rng.uniform(size=N)[:, None]**(1/3) * direction
    theta = np.arcsin(rng.uniform(-1, 1, N))
    phi = rng.uniform(0, 2*np.pi, N)
    return position, theta, phi


def radial_repulsion(position, velocity, a, width, strength, dt):
    """Velocity correction: damp outward radial motion, leave tangential motion intact.

    Exact exponential damping for a frozen wall normal/ramp over this step.
    """
    radius = np.linalg.norm(position, axis=1)
    normal = position / np.where(radius > 0, radius, 1)[:, None]
    ramp = np.clip((radius - a + width) / width, 0, 1)
    outward_speed = np.maximum(np.sum(velocity * normal, axis=1), 0.0)
    removed_fraction = -np.expm1(-strength * ramp * dt)
    return -(removed_fraction * outward_speed)[:, None] * normal


In [ ]:
def angles_from_velocity(velocity, old_phi):
    horizontal = np.hypot(velocity[:, 0], velocity[:, 1])
    theta = np.arctan2(velocity[:, 2], horizontal)
    phi = np.where(horizontal > 1e-12,
                   np.arctan2(velocity[:, 0], velocity[:, 1]), old_phi)
    return theta, phi % (2*np.pi)


def update_angles_sphere(position, theta, phi, R, eta, rng, dt):
    direction = update_velocity_3d(theta, phi, 1.0)
    displacement = position[:, None, :] - position[None, :, :]
    neighbors = np.sum(displacement**2, axis=2) <= R**2  # Euclidean; includes self.
    mean = neighbors @ direction
    lengths = np.linalg.norm(mean, axis=1)
    average = mean / np.where(lengths > 1e-12, lengths, 1)[:, None]
    average[lengths <= 1e-12] = direction[lengths <= 1e-12]
    noisy = add_direction_noise_3d(average, eta, rng, dt)
    return angles_from_velocity(noisy, phi)


In [ ]:
def confine_position(position, a):
    """Projected Euler boundary: keep every sample inside the closed ball."""
    radius = np.linalg.norm(position, axis=1)
    return position / np.maximum(1.0, radius/a)[:, None]


def tangent_wall_velocity(position, velocity, a):
    """Set v.n=0 at contact, preserving the tangential vector exactly."""
    result = np.array(velocity, dtype=float, copy=True)
    radius = np.linalg.norm(position, axis=1)
    contact = radius >= a * (1 - 1e-12)
    if np.any(contact):
        normal = position[contact] / radius[contact, None]
        radial_speed = np.sum(result[contact] * normal, axis=1)
        result[contact] -= radial_speed[:, None] * normal
    return result


def apply_wall_response(position, velocity, a, width, strength, dt):
    corrected = velocity + radial_repulsion(position, velocity, a, width, strength, dt)
    return tangent_wall_velocity(position, corrected, a)


In [ ]:
def vicsek_sphere(N, a, v_0, R, eta, *, wall_width=1.0, wall_strength=5.0,
                  dt=0.05, n_steps=800, seed=None):
    """Return x, v, theta, phi, t in a nonperiodic spherical container.

    x/v: (n_steps+1, N, 3); angles: (n_steps+1, N).
    v_0 is the preferred speed; wall damping/projection can reduce actual speed.
    Contact is a strict tangency constraint with time-step position projection.
    """
    if not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError("N must be a positive integer.")
    if not isinstance(n_steps, (int, np.integer)) or n_steps < 0:
        raise ValueError("n_steps must be a nonnegative integer.")
    if not np.isfinite([a, v_0, R, eta, wall_width, wall_strength, dt]).all():
        raise ValueError("Parameters must be finite.")
    if a <= 0 or dt <= 0 or not 0 < wall_width <= a:
        raise ValueError("a/dt must be positive and 0 < wall_width <= a.")
    if min(v_0, R, eta, wall_strength) < 0:
        raise ValueError("Speed, neighbor radius, noise and steering must be nonnegative.")
    rng = np.random.default_rng(seed)
    t = np.arange(n_steps + 1) * dt
    x = np.zeros((len(t), N, 3))
    v = np.zeros_like(x)
    theta = np.zeros((len(t), N))
    phi = np.zeros_like(theta)
    x[0], theta[0], phi[0] = initialize_sphere(N, a, rng)
    v[0] = apply_wall_response(x[0], update_velocity_3d(theta[0], phi[0], v_0),
                               a, wall_width, wall_strength, dt)
    initial_theta, initial_phi = angles_from_velocity(v[0], phi[0])
    moving = np.linalg.norm(v[0], axis=1) > 1e-12
    theta[0, moving], phi[0, moving] = initial_theta[moving], initial_phi[moving]
    for i in range(n_steps):
        theta_new, phi_new = update_angles_sphere(
            x[i], theta[i], phi[i], R, eta, rng, dt)
        x[i+1] = confine_position(x[i] + dt*v[i], a)
        candidate = update_velocity_3d(theta_new, phi_new, v_0)
        v[i+1] = apply_wall_response(x[i+1], candidate, a, wall_width, wall_strength, dt)
        theta[i+1], phi[i+1] = angles_from_velocity(v[i+1], phi_new)
        stopped = np.linalg.norm(v[i+1], axis=1) < 1e-12
        theta[i+1, stopped], phi[i+1, stopped] = theta_new[stopped], phi_new[stopped]
    return x, v, theta, phi, t


In [ ]:
x, v, theta, phi, t = vicsek_sphere(
    N, a, v_0, R, eta, wall_width=wall_width, wall_strength=wall_strength,
    dt=dt, n_steps=n_steps, seed=seed
)
radius = np.linalg.norm(x, axis=2)
radial_velocity = np.sum(x*v, axis=2) / np.where(radius > 0, radius, 1)
contact = radius >= a * (1 - 1e-12)
print("Maximum radius:", radius.max(), "/", a)
print("Maximum |v_r| at contact:", np.max(np.abs(radial_velocity[contact]), initial=0.0))
x.shape, v.shape, theta.shape, phi.shape, t.shape


In [ ]:
# Animation inside the sphere; arrows show the actual confined velocity.
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from IPython.display import HTML, Video, display
import imageio_ffmpeg

def arrow_segments_3d(position, direction, length):
    magnitude = np.linalg.norm(direction, axis=1)
    unit = direction / np.where(magnitude > 0, magnitude, 1)[:, None]
    reference = np.tile([0., 0., 1.], (len(position), 1))
    reference[np.abs(unit[:, 2]) > .9] = [1., 0., 0.]
    side = np.cross(reference, unit)
    side_length = np.linalg.norm(side, axis=1)
    side /= np.where(side_length > 0, side_length, 1)[:, None]
    scale = (length * magnitude)[:, None]
    tip = position + .65 * scale * unit
    tail = position - .35 * scale * unit
    wing = tip - .35 * scale * unit
    return np.concatenate([np.stack([tail, tip], axis=1),
                           np.stack([tip, wing + .18*scale*side], axis=1),
                           np.stack([tip, wing - .18*scale*side], axis=1)])


with plt.rc_context({"font.family": "DejaVu Sans", "animation.embed_limit": 80}):
    fig = plt.figure(figsize=(8, 7), dpi=85, facecolor="#080e1c")
    ax = fig.add_subplot(projection="3d", facecolor="#080e1c")
    longitude = np.linspace(0, 2*np.pi, 36)
    latitude = np.linspace(0, np.pi, 19)
    sphere_x = a*np.outer(np.cos(longitude), np.sin(latitude))
    sphere_y = a*np.outer(np.sin(longitude), np.sin(latitude))
    sphere_z = a*np.outer(np.ones_like(longitude), np.cos(latitude))
    ax.plot_wireframe(sphere_x, sphere_y, sphere_z, color="#6ce6ef", alpha=0.13, linewidth=0.5)
    ax.set(xlim=(-a,a), ylim=(-a,a), zlim=(-a,a), xlabel="x", ylabel="y", zlabel="z")
    ax.set_box_aspect((1,1,1))
    ax.view_init(elev=22, azim=-55)
    ax.tick_params(colors="#879bb5", labelsize=8)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color((0.05,0.09,0.16,1))
        axis.label.set_color("#b7c9df")
    direction = v / v_0 if v_0 > 0 else np.zeros_like(v)
    colors = plt.get_cmap("hsv")(phi[0]/(2*np.pi))
    arrows = Line3DCollection(arrow_segments_3d(x[0], direction[0], .12*a),
                              colors=np.tile(colors,(3,1)), linewidths=1.2)
    ax.add_collection3d(arrows)
    title = ax.set_title("", color="#edf5ff", fontsize=14)
    fps = 24
    frames = np.unique(np.linspace(0,len(t)-1,min(len(t),121)).astype(int))
    def update_sphere(i):
        arrows.set_segments(arrow_segments_3d(x[i],direction[i],.12*a))
        arrows.set_color(np.tile(plt.get_cmap("hsv")(phi[i]/(2*np.pi)),(3,1)))
        title.set_text(f"Vicsek inside a sphere | t={t[i]:.2f}\n"
                       f"wall contacts: {np.count_nonzero(contact[i])} / {N}")
        return arrows,title
    animation = FuncAnimation(fig,update_sphere,frames=frames,interval=1000/fps,blit=False)
    player = animation.to_jshtml(default_mode="loop")
    plt.close(fig)
display(HTML(player))


In [ ]:
# Run with Vicsek model as the working directory to save beside this notebook.
from pathlib import Path
from vicsek_sphere_player import make_vicsek_sphere_player

Path("vicsek_sphere.html").write_text(make_vicsek_sphere_player(
    N, a, v_0, R, eta, wall_width, wall_strength, dt, n_steps, seed
), encoding="utf-8")

with plt.rc_context({"animation.ffmpeg_path": imageio_ffmpeg.get_ffmpeg_exe()}):
    animation.save("vicsek_sphere.mp4", writer=FFMpegWriter(fps=fps), dpi=120)
display(Video("vicsek_sphere.mp4", embed=True))
